In [0]:
df = spark.table("metadata_governance.silver.silver_metadata_columns")
df.display()

In [0]:
from pyspark.sql.functions import col, when, expr

tier2_fields = ["column_desc", "table_desc", "data_steward", 
                 "security_classification", "term_subdomain", "certification_level"]

completeness_expr = " + ".join([f"CASE WHEN {f} IS NOT NULL THEN 1 ELSE 0 END" for f in tier2_fields])

column_detail = df.withColumn(
    "tier2_filled_count", expr(completeness_expr)
).withColumn(
    "row_completeness_pct", (col("tier2_filled_count") / len(tier2_fields)) * 100
).withColumn(
    "pii_non_compliant",
    when((col("pii_flag") == True) & (col("security_classification").isNull()), True).otherwise(False)
).withColumn(
    "unowned",
    when(col("data_steward").isNull(), True).otherwise(False)
).withColumn(
    "uncertified",
    when(col("certification_level").isNull(), True).otherwise(False)
).select(
    "column_id", "column_name", "table_id", "table_name", "schema_name",
    "database_name", "system_name", "pii_flag", "critical_data_element_flag",
    "row_completeness_pct", "pii_non_compliant", "unowned", "uncertified"
)

column_detail.display()

In [0]:
column_detail.write.mode("overwrite").saveAsTable("metadata_governance.gold.column_governance_detail")

In [0]:
from pyspark.sql.functions import avg, sum as spark_sum, count, round as spark_round

table_summary = column_detail.groupBy("table_id", "table_name", "schema_name", "database_name", "system_name").agg(
    spark_round(avg("row_completeness_pct"), 2).alias("table_completeness_pct"),
    count("column_id").alias("total_columns"),
    spark_sum(col("pii_non_compliant").cast("int")).alias("pii_non_compliant_count"),
    spark_sum(col("unowned").cast("int")).alias("unowned_count"),
    spark_sum(col("uncertified").cast("int")).alias("uncertified_count")
).withColumn(
    "maturity_tier",
    when(col("table_completeness_pct") >= 90, "High")
    .when(col("table_completeness_pct") >= 50, "Medium")
    .otherwise("Low")
)

table_summary.display()

In [0]:
table_summary.write.mode("overwrite").saveAsTable("metadata_governance.gold.table_governance_summary")

In [0]:
spark.sql("SELECT * FROM metadata_governance.gold.table_governance_summary LIMIT 10").display()

In [0]:
spark.sql("SELECT * FROM metadata_governance.gold.column_governance_detail LIMIT 10").display()